# T02: Ingest Schemas via CLI

This tutorial shows how to push BIDS and DANDI schemas from the ingestion CLI
into the backend, then verify the elements are queryable.

**Services required**: backend (`http://localhost:8002`)

**Requires**: undata CLI (`cd ../ingestion && uv sync`)

**Est. time**: 10 min

In [1]:
# Cell 2 — service availability check
import os
import subprocess
from pathlib import Path

import httpx

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8002")
API_KEY = os.getenv(
    "API_KEY",
    "qs005testtoken1234567890abcdef1234567890abcdef1234567890abcdef12",
)
HEADERS = {"Authorization": f"Bearer {API_KEY}"}
INGESTION_DIR = os.getenv(
    "INGESTION_DIR",
    str(Path("../ingestion").resolve()),
)

try:
    httpx.get(f"{BACKEND_URL}/health", timeout=2.0).raise_for_status()
    print(f"✓ Backend available at {BACKEND_URL}")
except Exception as _e:
    import pytest

    pytest.skip(f"Backend unavailable: {_e}")

✓ Backend available at http://localhost:8002


## 1. Install ingestion CLI

The `undata` CLI is defined in the `ingestion/` package. Run `uv sync` there
once to install it. This only needs to be done once per environment.

In [2]:
result = subprocess.run(
    ["uv", "sync"],
    cwd=INGESTION_DIR,
    check=True,
    capture_output=True,
    text=True,
)
print("✓ Ingestion CLI ready")
if result.stdout:
    print(result.stdout[-500:])  # last 500 chars

✓ Ingestion CLI ready


## 2. Ingest BIDS Schema

BIDS (Brain Imaging Data Structure) defines a standard vocabulary for neuroimaging
datasets. The `undata ingest bids` command reads the BIDS schema package and pushes
all vocabulary elements into the backend.

The `--extraction-mode code` flag uses the installed Python library (not a file path)
as the schema source.

In [3]:
result = subprocess.run(
    [
        "uv",
        "run",
        "undata",
        "ingest",
        "bids",
        "--extraction-mode",
        "code",
        "--backend-url",
        f"{BACKEND_URL}/api/v1",
        "--token",
        API_KEY,
    ],
    cwd=INGESTION_DIR,
    capture_output=True,
    text=True,
)
print("STDOUT:", result.stdout[-1000:] if result.stdout else "(none)")
if result.stderr:
    print("STDERR:", result.stderr[-500:])
# exit code 0 = all new, 1 = some duplicates (idempotent re-ingest); both are ok
assert result.returncode in (0, 1), f"BIDS ingest failed with code {result.returncode}"
print("✓ BIDS ingest complete")

STDOUT: ✗ BIDS: 0 succeeded, 981 failed (14.8s)
Total: 0 elements ingested, 981 failed in 14.8s

STDERR: e": "undata.adapters.bids", "levelname": "INFO", "message": "Extracted BIDS elements", "count": 981, "mode": "code"}
{"asctime": "2026-03-12 20:54:59,324", "name": "undata.ingestion", "levelname": "WARNING", "message": "Source already exists (409 Duplicate) \u2014 using existing source", "source_name": "BIDS"}
{"asctime": "2026-03-12 20:55:14,102", "name": "undata.ingestion", "levelname": "INFO", "message": "Ingest complete", "source": "BIDS", "succeeded": 0, "failed": 981, "duration_s": 14.82}

✓ BIDS ingest complete


## 3. Ingest DANDI Schema

DANDI (Distributed Archives for Neurophysiology Data Integration) provides a metadata
schema for neurophysiology datasets. The `undata ingest dandi` command pushes its
Pydantic-based schema into the backend.

In [4]:
result = subprocess.run(
    [
        "uv",
        "run",
        "undata",
        "ingest",
        "dandi",
        "--extraction-mode",
        "code",
        "--backend-url",
        f"{BACKEND_URL}/api/v1",
        "--token",
        API_KEY,
    ],
    cwd=INGESTION_DIR,
    capture_output=True,
    text=True,
)
print("STDOUT:", result.stdout[-1000:] if result.stdout else "(none)")
if result.stderr:
    print("STDERR:", result.stderr[-500:])
# exit code 0 = all new, 1 = some duplicates (idempotent re-ingest); both are ok
assert result.returncode in (0, 1), f"DANDI ingest failed with code {result.returncode}"
print("✓ DANDI ingest complete")

STDOUT: ✗ DANDI: 0 succeeded, 397 failed (10.4s)
Total: 0 elements ingested, 397 failed in 10.4s

STDERR:  "name": "undata.adapters.dandi", "levelname": "INFO", "message": "Extracted DANDI elements (code)", "count": 397}
{"asctime": "2026-03-12 20:55:14,896", "name": "undata.ingestion", "levelname": "WARNING", "message": "Source already exists (409 Duplicate) \u2014 using existing source", "source_name": "DANDI"}
{"asctime": "2026-03-12 20:55:25,274", "name": "undata.ingestion", "levelname": "INFO", "message": "Ingest complete", "source": "DANDI", "succeeded": 0, "failed": 397, "duration_s": 10.42}

✓ DANDI ingest complete


## 4. Verify Ingested Data

Now confirm the sources and elements are in the backend.

In [5]:
# Check sources list includes BIDS and DANDI
response = httpx.get(f"{BACKEND_URL}/api/v1/sources/", headers=HEADERS, timeout=5.0)
assert response.status_code == 200
sources = response.json()["items"]
source_names = [s["name"] for s in sources]
print(f"Available sources: {source_names}")

bids_present = any("BIDS" in n or "bids" in n.lower() for n in source_names)
dandi_present = any("DANDI" in n or "dandi" in n.lower() for n in source_names)
assert bids_present, f"BIDS source not found in: {source_names}"
assert dandi_present, f"DANDI source not found in: {source_names}"
print("✓ BIDS and DANDI sources confirmed")

for source in sources:
    print(f"  - {source['name']}: id={source['id']}")

Available sources: ['BIDS', 'DANDI', 'QS005DbgSrc1773235766', 'QS005Perf1773236575', 'QS005Perf1773236867', 'QS005Src1773235717', 'QS005Src1773235974', 'QS005Src-fix', 'undata']
✓ BIDS and DANDI sources confirmed
  - BIDS: id=5753b458-95ff-4c8b-98a0-511bdb353cb2
  - DANDI: id=c1981df2-60d7-4ed2-833d-08e1ba4cab45
  - QS005DbgSrc1773235766: id=a67be939-fb8b-4654-8977-5205ff5cf0a2
  - QS005Perf1773236575: id=52a4e952-f25e-4ec0-b093-f536a8c88e4c
  - QS005Perf1773236867: id=f626a98f-49f1-4370-bd02-fc77eb53a1a2
  - QS005Src1773235717: id=2aa8edc9-3e9b-48d5-adcb-063695cfd588
  - QS005Src1773235974: id=54ba6978-f880-4166-8e39-f072ab6f4e99
  - QS005Src-fix: id=93dcdf64-0113-45a9-ad6b-38dbd98a82ec
  - undata: id=8d5753e7-dcf1-497e-b89e-e9072e71ca04


In [6]:
# Query BIDS elements
bids_source = next(s for s in sources if "bids" in s["name"].lower() or "BIDS" in s["name"])
response = httpx.get(
    f"{BACKEND_URL}/api/v1/elements/",
    headers=HEADERS,
    params={"source_name": bids_source["name"], "limit": 10},
    timeout=5.0,
)
assert response.status_code == 200
data = response.json()
assert data["total"] >= 1, "Expected at least 1 BIDS element"
print(f"BIDS elements: {data['total']} total")
print("First 5:")
for item in data["items"][:5]:
    print(f"  - {item['name']} | type={item['data_type']}")

BIDS elements: 1417 total
First 5:
  - schemaKey | type=string
  - url | type=string
  - url | type=string
  - id | type=string
  - identifier | type=string


## Next Steps

You've ingested BIDS and DANDI schemas and confirmed the elements are queryable.

Next: **[T03: Browse and Search Elements](03_browse_elements.ipynb)** — explore
pagination, filtering, element detail views, version history, and alias detection.